In [3]:
from agents import (
    Agent,
    Runner,
    GuardrailFunctionOutput,
    RunContextWrapper,
    TResponseInputItem,
    input_guardrail,
    InputGuardrailTripwireTriggered,
    output_guardrail,
    OutputGuardrailTripwireTriggered,
    handoffs,
    handoff,
)

from pydantic import BaseModel
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

In [3]:
agent = Agent(
    name="test study assistant",
    model="gpt-4o-mini",
    instructions="You are a helpful study assistant.",
)

convo: list[TResponseInputItem] = []

while True:
    user_input = input("User : ")
    print("User : ", user_input)

    if "exit" in user_input:
        break

    convo.append({"content": user_input, "role": "user"})
    result = await Runner.run(agent, convo)
    print(f"Agent : {result.final_output}")

    convo = result.to_input_list()

User :  hei
Agent : Hei! Hvordan kan jeg hjelpe deg i dag?
User :  wut
Agent : Looks like you might be surprised! How can I assist you?
User :  what was that language before
Agent : That was Norwegian! "Hei" means "hello." If you have any specific questions or topics you'd like to discuss, feel free to let me know!
User :  exit


In [10]:
import asyncio
from agents import RunContextWrapper
from pydantic import BaseModel


class Quote(BaseModel):
    personality_name: str
    quote_attributed: str
    language: str


translator_agent_arabic = Agent(
    name="Arabic Translator Agent",
    handoff_description="Translates the given text in Arabic",
    instructions="""You are a translator agent. You'll be given a quote as {"Name": "<the quote of him>"} and your job is to translate the whole text into Arabic language. and give the final output as {"Name": "<the translated quote>"}.""",
    model="gpt-4o-mini",
    output_type=Quote,
)


translator_agent_urdu = Agent(
    name="Urdu Translator Agent",
    handoff_description="Translates the given text in Urdu",
    instructions="""You are a translator agent. You'll be given a quote as {"Name": "<the quote of him>"} and your job is to translate the whole text into Urdu language. and give the final output as {"Name": "<the translated quote>"}.""",
    model="gpt-4o-mini",
    output_type=Quote,
)

translator_agent_english = Agent(
    name="English Translator Agent",
    handoff_description="Translates the given text in English",
    instructions="""You are a translator agent. You'll be given a quote as {"Name": "<the quote of him>"} and your job is to translate the whole text into English language. and give the final output as {"Name": "<the translated quote>"}.""",
    model="gpt-4o-mini",
    output_type=Quote,
)


def on_urdu_handoff(ctx: RunContextWrapper[None]):
    print("Handing off to Urdu translator agent")


def on_arabic_handoff(ctx: RunContextWrapper[None]):
    print("Handing off to Arabic translator agent")


def on_english_handoff(ctx: RunContextWrapper[None]):
    print("Handing off to English translator agent")


quote_agent = Agent(
    name="Quote Agent",
    instructions="""
    You are a quote agent. You'll be given a person name and a language name in which it is required in. 
        Find one relevant quote attributed to that name. and hand it to the required translator agent
        if the language is 'Arabic', always hand off to the Arabic translator agent.
        If the language is 'Urdu', always hand off to the Urdu translator agent.
        If the language is 'English', always hand off to the English translator agent.
    """,
    model="gpt-4o-mini",
    output_type=Quote,
    handoffs=[
        handoff(
            agent=translator_agent_arabic,
            on_handoff=(on_arabic_handoff),
        ),
        handoff(
            agent=translator_agent_urdu,
            on_handoff=(on_urdu_handoff),
        ),
        handoff(agent=translator_agent_english, on_handoff=(on_english_handoff)),
    ],
)
data = []
convo: list[TResponseInputItem] = []

while True:
    person = input("Enter a personality name to get a quote(or exit using 'quit') ")

    if person.lower() == "quit":
        break
    language = (
        input("Enter the language (arabic/urdu) for translation: ").strip().lower()
    )

    quote = await Runner.run(
        quote_agent, "Give me " + person + "'s quote in " + language + " language"
    )
    print(quote.final_output)

    quote.to_input_list()

Handing off to English translator agent
personality_name='Arthur Schopenhauer' quote_attributed='"Compassion is the basis of morality."' language='English'
Handing off to Arabic translator agent
personality_name='علي بن أبي طالب (ع)' quote_attributed='"الناس عبيد الدنيا، كما أن عبيد الدنيا، فإنهم عبيد الفرح، فإنهم يستوحشون في الوقت الذي يتركون فيه الفرح."' language='Arabic'
Handing off to Urdu translator agent
personality_name='اقبال' quote_attributed='خودی کو کر بلند اتنا کہ ہر تقدیر سے پہلے، خدا بندے سے خود پوچھے، بتا تیری رضا کیا ہے۔' language='Urdu'
Handing off to English translator agent
personality_name='Fyodor Dostoevsky' quote_attributed='"The mystery of human existence lies not in just staying alive, but in finding something to live for."' language='English'
Handing off to English translator agent
personality_name='Victor Frankl' quote_attributed='When we are no longer able to change a situation, we are challenged to change ourselves.' language='English'
Handing off to English